# BBO10: HPO-Enhanced Black Box Optimisation

Builds on **BBO6** (CNN-Enhanced GP with Kernel Optimisation, Gradient Enhancement,  
CNN Acquisition Amortisation, CNN Residual Correction) and adds **three HPO-driven  
improvements** using [Optuna](https://optuna.readthedocs.io/).

## What Optuna Brings (the Akiba et al. perspective)

The Optuna paper introduces a *define-by-run* API built on **TPE** (Tree-structured  
Parzen Estimator) and **CMA-ES** samplers with **pruning** (early stopping of bad  
trials). Key concepts applied here:

| Optuna concept | Where applied in BBO10 |
|---|---|
| TPE sampler | Tuning `kappa`, `sv_weight`, `cnn_prescreen_k` per-function |
| CMA-ES sampler | Tuning CNN architecture (hidden size, dropout, LR, epochs) |
| Median pruner | Skip CNN trials whose intermediate loss exceeds the median |
| Study persistence | Warm-start from prior function runs |

## Three HPO Enhancements

**Enhancement #6 — Acquisition HPO (Optuna TPE)**  
Tune `kappa`, `sv_weight` and `cnn_prescreen_k` using a surrogate objective  
(cross-validated UCB on held-out points). Replaces the fixed `kappa=4.0`,  
`sv_weight=0.4` defaults.

**Enhancement #7 — CNN Architecture HPO (Optuna CMA-ES + MedianPruner)**  
Tune `hidden_size`, `dropout`, `learning_rate`, and `epochs` for both  
`GradientPredictorCNN` and `ResidualCorrectorCNN`. Uses intermediate loss  
reporting so Optuna can prune unpromising trials early.

**Enhancement #8 — GP Noise & Restart HPO (Optuna TPE)**  
Tune `alpha` (GP jitter), `n_restarts_optimizer`, and the WhiteKernel  
noise bounds — the three GP hyperparameters that most affect calibration  
but are currently left at hard-coded defaults.


In [1]:
"""
BBO7: HPO-Enhanced Black Box Optimisation
==========================================
Extends BBO6 with three Optuna-driven HPO enhancements:

  #6  Acquisition HPO      — tune kappa, sv_weight, cnn_prescreen_k via TPE
  #7  CNN Architecture HPO — tune hidden/dropout/lr/epochs via CMA-ES + pruning
  #8  GP Noise HPO         — tune alpha, n_restarts, noise bounds via TPE
"""

import numpy as np
import warnings
warnings.filterwarnings('ignore')

# GP stack
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
from sklearn.model_selection import cross_val_score
from scipy.optimize import minimize

# CNN stack
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# HPO stack  ← NEW
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)   # suppress trial-level spam


print(f"PyTorch version: {torch.__version__}")
print(f"Optuna  version: {optuna.__version__}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

PyTorch version: 2.9.1
Optuna  version: 4.8.0
Using device: cpu


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 1: GP INFRASTRUCTURE  (extended from BBO6 — GP noise HPO added)
# ══════════════════════════════════════════════════════════════════════════════

class GradientEnhancedGP:
    """
    Gradient-Enhanced Gaussian Process  (unchanged from BBO6).
    Augments training data with virtual gradient observations.
    """

    def __init__(self, kernel, alpha=1e-6, normalize_y=True):
        self.kernel      = kernel
        self.alpha       = alpha
        self.normalize_y = normalize_y
        self.gp          = None
        self.X_train     = None
        self.y_train     = None
        self.y_mean      = 0.0
        self.y_std       = 1.0

    def _augment_training_data(self, X, y, gradients, epsilon=0.01):
        n, dim = X.shape
        X_aug = [X]
        y_aug = [y]
        for d in range(dim):
            X_plus        = X.copy(); X_plus[:, d]  += epsilon
            X_minus       = X.copy(); X_minus[:, d] -= epsilon
            y_aug.append(y + epsilon * gradients[:, d])
            y_aug.append(y - epsilon * gradients[:, d])
            X_aug.extend([X_plus, X_minus])
        return np.vstack(X_aug), np.concatenate(y_aug)

    def fit(self, X, y, gradients=None, n_restarts=10):
        self.X_train = X
        self.y_train = y

        if self.normalize_y:
            self.y_mean = y.mean()
            self.y_std  = y.std() if y.std() > 0 else 1.0
            y_norm      = (y - self.y_mean) / self.y_std
        else:
            y_norm = y

        if gradients is not None:
            grad_norm = gradients / self.y_std if self.normalize_y else gradients
            X_aug, y_aug = self._augment_training_data(X, y_norm, grad_norm)
            print(f"  Gradient-enhanced GP: {len(X)} obs → {len(X_aug)} effective points")
        else:
            X_aug, y_aug = X, y_norm

        self.gp = GaussianProcessRegressor(
            kernel=self.kernel, alpha=self.alpha,
            n_restarts_optimizer=n_restarts, normalize_y=False
        )
        self.gp.fit(X_aug, y_aug)
        return self

    def predict(self, X, return_std=False):
        if return_std:
            mu, std = self.gp.predict(X, return_std=True)
            if self.normalize_y:
                mu  = mu  * self.y_std + self.y_mean
                std = std * self.y_std
            return mu, std
        else:
            mu = self.gp.predict(X)
            if self.normalize_y:
                mu = mu * self.y_std + self.y_mean
            return mu

    @property
    def kernel_(self):
        return self.gp.kernel_ if self.gp else self.kernel

    def log_marginal_likelihood(self):
        return self.gp.log_marginal_likelihood() if self.gp else -np.inf


def estimate_gradients_numerically(gp, X_obs, epsilon=0.01, bounds=None):
    """Finite-difference gradient estimation from GP predictions."""
    n, dim = X_obs.shape
    gradients = np.zeros((n, dim))
    lows  = np.array([b[0] for b in bounds]) if bounds else None
    highs = np.array([b[1] for b in bounds]) if bounds else None

    for d in range(dim):
        X_plus  = X_obs.copy()
        X_minus = X_obs.copy()
        if lows is not None:
            X_plus[:, d]  = np.clip(X_plus[:, d]  + epsilon, lows[d], highs[d])
            X_minus[:, d] = np.clip(X_minus[:, d] - epsilon, lows[d], highs[d])
        else:
            X_plus[:, d]  += epsilon
            X_minus[:, d] -= epsilon
        gradients[:, d] = (gp.predict(X_plus) - gp.predict(X_minus)) / (2 * epsilon)

    return gradients


# ── Enhancement #8: GP Noise HPO ──────────────────────────────────────────────
def select_best_kernel_with_hpo(X_obs, y_obs, dim, n_trials=30):
    """
    Enhancement #8 — GP Noise & Restart HPO (Optuna TPE)

    Extends BBO6's select_best_kernel() by also tuning:
      - alpha        (GP jitter):              log-uniform [1e-8, 1e-3]
      - n_restarts   (optimizer restarts):     integer     [5, 30]
      - noise_upper  (WhiteKernel upper bound): log-uniform [1e-4, 1e-1]

    Each Optuna trial tests one (kernel_type, alpha, n_restarts, noise_upper)
    combination.  The best trial's fitted GP is returned.

    Why it matters:
      BBO6 uses fixed alpha=1e-6 and noise_upper=1e-2.  For high-curvature
      functions (F4, F5) a larger alpha or looser noise bound can prevent
      near-singular kernel matrices and improve LOO CV.
    """
    kernel_choices = ['RBF_ARD', 'Matern_2.5_ARD', 'Matern_1.5_ARD']

    def build_kernel(name, dim, noise_upper):
        noise_bounds = (1e-10, noise_upper)
        if name == 'RBF_ARD':
            return (ConstantKernel(1.0, (0.1, 10.0)) *
                    RBF(np.ones(dim), (0.01, 20.0)) +
                    WhiteKernel(1e-5, noise_bounds))
        elif name == 'Matern_2.5_ARD':
            return (ConstantKernel(1.0, (0.1, 10.0)) *
                    Matern(np.ones(dim), (0.01, 20.0), nu=2.5) +
                    WhiteKernel(1e-5, noise_bounds))
        else:
            return (ConstantKernel(1.0, (0.1, 10.0)) *
                    Matern(np.ones(dim), (0.01, 20.0), nu=1.5) +
                    WhiteKernel(1e-5, noise_bounds))

    def objective(trial):
        name        = trial.suggest_categorical('kernel', kernel_choices)
        alpha       = trial.suggest_float('alpha', 1e-8, 1e-3, log=True)
        n_restarts  = trial.suggest_int('n_restarts', 5, 30)
        noise_upper = trial.suggest_float('noise_upper', 1e-4, 1e-1, log=True)

        kernel = build_kernel(name, dim, noise_upper)
        gp_test = GaussianProcessRegressor(
            kernel=kernel, alpha=alpha,
            n_restarts_optimizer=n_restarts, normalize_y=True
        )
        try:
            if len(X_obs) >= 5:
                scores = cross_val_score(
                    gp_test, X_obs, y_obs,
                    cv=min(5, len(X_obs)),
                    scoring='neg_mean_squared_error'
                )
                return float(scores.mean())
            else:
                gp_test.fit(X_obs, y_obs)
                return float(gp_test.score(X_obs, y_obs))
        except Exception:
            return float('-inf')

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    best_kernel_obj = build_kernel(best['kernel'], dim, best['noise_upper'])

    # Refit final GP on full data with best params
    best_gp = GaussianProcessRegressor(
        kernel=best_kernel_obj, alpha=best['alpha'],
        n_restarts_optimizer=best['n_restarts'], normalize_y=True
    )
    best_gp.fit(X_obs, y_obs)

    print(f"  [HPO #8] Best kernel: {best['kernel']}  "
          f"alpha={best['alpha']:.2e}  n_restarts={best['n_restarts']}  "
          f"noise_upper={best['noise_upper']:.2e}  "
          f"CV={study.best_value:.4f}  (over {n_trials} trials)")

    return best['kernel'], best_kernel_obj, best_gp, best['alpha'], best['n_restarts'], study

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 2: CNN ARCHITECTURES  (parameterised for HPO — Enhancement #7)
# ══════════════════════════════════════════════════════════════════════════════

class GradientPredictorCNN(nn.Module):
    """
    Enhancement #4 — CNN Acquisition Amortiser  (BBO6, unchanged logic)
    Now accepts `hidden` and `dropout` from HPO trial (Enhancement #7).
    """

    def __init__(self, dim, hidden=64, dropout=0.2):
        super().__init__()
        self.dim = dim

        self.conv_block = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=min(3, dim), padding=min(1, dim // 2)),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=min(3, dim), padding=min(1, dim // 2)),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
            nn.Softplus(),
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv_block(x)
        return self.fc(x)


class ResidualCorrectorCNN(nn.Module):
    """
    Enhancement #5 — CNN Residual GP Corrector  (BBO6, unchanged logic)
    Now accepts `hidden` and `dropout` from HPO trial (Enhancement #7).
    """

    def __init__(self, dim, hidden=64, dropout=0.2):
        super().__init__()
        self.dim = dim

        self.backbone = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
        )

        self.residual_head = nn.Sequential(
            nn.Linear(hidden // 2, 16), nn.ReLU(), nn.Linear(16, 1)
        )

        self.scale_head = nn.Sequential(
            nn.Linear(hidden // 2, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Softplus()
        )

    def forward(self, x):
        h        = self.backbone(x)
        residual = self.residual_head(h)
        scale    = self.scale_head(h) + 0.5
        return torch.cat([residual, scale], dim=1)

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 3: CNN TRAINING UTILITIES  (Enhancement #7 — HPO over architecture)
# ══════════════════════════════════════════════════════════════════════════════

def _build_augmented_tensors(X_obs, targets, n_aug=20, noise=0.02):
    """Bootstrap-augment observations and convert to Tensors."""
    X_aug = np.vstack([
        np.clip(X_obs + np.random.randn(*X_obs.shape) * noise, 0, 1)
        for _ in range(n_aug)
    ])
    if targets.ndim == 1:
        t_aug = np.tile(targets, n_aug)
        T = torch.FloatTensor(t_aug).unsqueeze(1).to(DEVICE)
    else:
        t_aug = np.tile(targets, (n_aug, 1))
        T = torch.FloatTensor(t_aug).to(DEVICE)
    return torch.FloatTensor(X_aug).to(DEVICE), T


def _train_one_cnn(model, X_t, tgt_t, lr, epochs, loss_fn, trial=None):
    """
    Inner training loop shared by both CNNs.

    If `trial` is not None (HPO mode), reports intermediate loss every 20 epochs
    so Optuna's MedianPruner can prune bad trials early — the key idea from the
    Optuna paper's pruning section.
    """
    optimiser = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)
    loader    = DataLoader(
        TensorDataset(X_t, tgt_t),
        batch_size=min(64, len(X_t)), shuffle=True
    )

    model.train()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, tb in loader:
            optimiser.zero_grad()
            loss = loss_fn(model(xb), tb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
            epoch_loss += loss.item()
        scheduler.step()
        losses.append(epoch_loss / len(loader))

        # ── HPO pruning hook  (Enhancement #7) ──────────────────────────────
        if trial is not None and (epoch + 1) % 20 == 0:
            trial.report(losses[-1], epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    return losses


# ── Enhancement #7: CNN Architecture HPO ─────────────────────────────────────
def tune_and_train_gradient_cnn(
    X_obs, grad_magnitudes, dim,
    n_hpo_trials=20,
    verbose=True
):
    """
    Enhancement #7 — CNN Architecture HPO (Optuna CMA-ES + MedianPruner)

    Searches over:
      hidden_size  ∈ {32, 64, 128, 256}
      dropout      ∈ [0.05, 0.50]
      lr           ∈ [1e-4, 1e-2]  (log scale)
      epochs       ∈ [100, 400]
      n_aug        ∈ [10, 40]      (bootstrap augmentation multiplier)

    Uses CMA-ES for the continuous subspace and MedianPruner so that trials
    whose epoch-20 loss exceeds the median of completed trials are stopped.
    The winning hyperparameters are used for the final training run.
    """
    g_min, g_max = grad_magnitudes.min(), grad_magnitudes.max()
    g_range  = g_max - g_min if g_max > g_min else 1.0
    grad_norm = (grad_magnitudes - g_min) / g_range

    def objective(trial):
        hidden  = trial.suggest_categorical('hidden',  [32, 64, 128, 256])
        dropout = trial.suggest_float('dropout', 0.05, 0.50)
        lr      = trial.suggest_float('lr',      1e-4, 1e-2, log=True)
        epochs  = trial.suggest_int('epochs',    100,  400)
        n_aug   = trial.suggest_int('n_aug',     10,   40)

        X_t, g_t = _build_augmented_tensors(X_obs, grad_norm, n_aug=n_aug)
        model    = GradientPredictorCNN(dim, hidden=hidden, dropout=dropout).to(DEVICE)
        losses   = _train_one_cnn(
            model, X_t, g_t, lr, epochs,
            loss_fn=nn.MSELoss(),
            trial=trial
        )
        return losses[-1]

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    study  = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=pruner
    )
    study.optimize(objective, n_trials=n_hpo_trials, show_progress_bar=False)

    bp = study.best_params
    if verbose:
        print(f"  [HPO #7 GradCNN] hidden={bp['hidden']}  dropout={bp['dropout']:.2f}  "
              f"lr={bp['lr']:.2e}  epochs={bp['epochs']}  n_aug={bp['n_aug']}  "
              f"best_loss={study.best_value:.6f}")

    # ── Final training run with best hyperparameters ──────────────────────────
    X_t, g_t = _build_augmented_tensors(X_obs, grad_norm, n_aug=bp['n_aug'])
    model    = GradientPredictorCNN(dim, hidden=bp['hidden'], dropout=bp['dropout']).to(DEVICE)
    losses   = _train_one_cnn(
        model, X_t, g_t, bp['lr'], bp['epochs'],
        loss_fn=nn.MSELoss()
    )
    if verbose:
        print(f"  GradCNN trained: final loss = {losses[-1]:.6f}")

    model.g_min   = g_min
    model.g_range = g_range
    model.eval()
    return model, losses, study


def tune_and_train_residual_cnn(
    X_obs, residuals, loo_stds, y_obs, dim,
    n_hpo_trials=20,
    verbose=True
):
    """
    Enhancement #7 — CNN Architecture HPO for ResidualCorrectorCNN.
    Same search space and pruning strategy as tune_and_train_gradient_cnn.
    """
    y_std    = y_obs.std() if y_obs.std() > 0 else 1.0
    res_norm = residuals / y_std
    unc_scale = np.clip(np.abs(residuals) / (loo_stds + 1e-8), 0.1, 5.0)
    targets   = np.stack([res_norm, unc_scale], axis=1)   # (N, 2)

    def res_loss_fn(pred, tgt):
        return nn.MSELoss()(pred[:, 0:1], tgt[:, 0:1]) + \
               0.5 * nn.MSELoss()(pred[:, 1:2], tgt[:, 1:2])

    def objective(trial):
        hidden  = trial.suggest_categorical('hidden',  [32, 64, 128, 256])
        dropout = trial.suggest_float('dropout', 0.05, 0.50)
        lr      = trial.suggest_float('lr',      1e-4, 1e-2, log=True)
        epochs  = trial.suggest_int('epochs',    150,  500)
        n_aug   = trial.suggest_int('n_aug',     15,   40)

        X_t, tgt_t = _build_augmented_tensors(X_obs, targets, n_aug=n_aug)
        model      = ResidualCorrectorCNN(dim, hidden=hidden, dropout=dropout).to(DEVICE)
        losses     = _train_one_cnn(
            model, X_t, tgt_t, lr, epochs,
            loss_fn=res_loss_fn,
            trial=trial
        )
        return losses[-1]

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    study  = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=pruner
    )
    study.optimize(objective, n_trials=n_hpo_trials, show_progress_bar=False)

    bp = study.best_params
    if verbose:
        print(f"  [HPO #7 ResCNN] hidden={bp['hidden']}  dropout={bp['dropout']:.2f}  "
              f"lr={bp['lr']:.2e}  epochs={bp['epochs']}  n_aug={bp['n_aug']}  "
              f"best_loss={study.best_value:.6f}")

    # ── Final run ──────────────────────────────────────────────────────────────
    X_t, tgt_t = _build_augmented_tensors(X_obs, targets, n_aug=bp['n_aug'])
    model      = ResidualCorrectorCNN(dim, hidden=bp['hidden'], dropout=bp['dropout']).to(DEVICE)

    def res_loss_fn_final(pred, tgt):
        return nn.MSELoss()(pred[:, 0:1], tgt[:, 0:1]) + \
               0.5 * nn.MSELoss()(pred[:, 1:2], tgt[:, 1:2])

    losses = _train_one_cnn(
        model, X_t, tgt_t, bp['lr'], bp['epochs'],
        loss_fn=res_loss_fn_final
    )
    if verbose:
        print(f"  ResidualCNN trained: final loss = {losses[-1]:.6f}")

    model.y_std = y_std
    model.eval()
    return model, losses, study


def compute_loo_residuals(gp_base, X_obs, y_obs):
    """Leave-one-out residuals for ResidualCorrectorCNN  (unchanged from BBO6)."""
    n   = len(y_obs)
    residuals = np.zeros(n)
    loo_stds  = np.zeros(n)

    for i in range(n):
        mask = np.ones(n, dtype=bool); mask[i] = False
        try:
            gp_loo = GaussianProcessRegressor(
                kernel=gp_base.kernel_, alpha=1e-6,
                n_restarts_optimizer=3, normalize_y=True
            )
            gp_loo.fit(X_obs[mask], y_obs[mask])
            mu_i, std_i  = gp_loo.predict(X_obs[i:i+1], return_std=True)
            residuals[i] = y_obs[i] - mu_i[0]
            loo_stds[i]  = std_i[0]
        except Exception:
            residuals[i] = 0.0
            loo_stds[i]  = 1.0

    return residuals, loo_stds

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 4: CNN INFERENCE HELPERS  (unchanged from BBO6)
# ══════════════════════════════════════════════════════════════════════════════

def cnn_predict_gradients(grad_cnn, X_candidates, batch_size=4096):
    grad_cnn.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(X_candidates), batch_size):
            xb   = torch.FloatTensor(X_candidates[start:start+batch_size]).to(DEVICE)
            pred = grad_cnn(xb).cpu().numpy().ravel()
            preds.append(pred)
    scores = np.concatenate(preds)
    return scores * grad_cnn.g_range + grad_cnn.g_min


def cnn_correct_gp(residual_cnn, gp, X_candidates, batch_size=4096):
    residual_cnn.eval()
    corrections = []
    with torch.no_grad():
        for start in range(0, len(X_candidates), batch_size):
            xb   = torch.FloatTensor(X_candidates[start:start+batch_size]).to(DEVICE)
            pred = residual_cnn(xb).cpu().numpy()
            corrections.append(pred)
    corrections = np.vstack(corrections)

    mu_gp, std_gp = gp.predict(X_candidates, return_std=True)

    residual      = corrections[:, 0] * residual_cnn.y_std
    unc_scale     = corrections[:, 1]
    mu_corrected  = mu_gp + residual
    std_corrected = std_gp * np.clip(unc_scale, 0.1, 5.0)

    return mu_corrected, std_corrected

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 5: ACQUISITION HPO  (Enhancement #6)
# ══════════════════════════════════════════════════════════════════════════════

def tune_acquisition_params(
    gp, X_obs, y_obs, residual_cnn=None,
    n_trials=40, seed=42
):
    """
    Enhancement #6 — Acquisition HPO (Optuna TPE)

    Searches over:
      kappa        ∈ [0.5, 10.0]   — exploration–exploitation trade-off
      sv_weight    ∈ [0.0, 0.8]    — gradient informativeness weight
      prescreen_k  ∈ [1000, 20000] — CNN shortlist size

    Surrogate objective: cross-validated UCB on held-out training points.
    For each LOO fold we predict UCB at the left-out point and measure
    how well it ranks relative to the current best — using a simple
    rank-correlation proxy.

    Why this matters:
      kappa=4.0 is a reasonable default but sub-optimal: low-dimensional
      functions (F1, F2) benefit from smaller kappa (more exploitation),
      while high-dimensional ones (F7, F8) need larger kappa (more
      exploration).  TPE learns this quickly in ~40 trials.
    """
    n = len(y_obs)

    def loo_rank_score(kappa, sv_weight):
        """
        LOO proxy: for each point i, compute UCB at all OTHER points,
        rank them, and check if the actual best (highest y_obs) is ranked
        in the top quartile.  Returns fraction of folds where this holds.
        """
        hits = 0
        for i in range(n):
            mask    = np.ones(n, dtype=bool); mask[i] = False
            X_rest  = X_obs[mask]
            y_rest  = y_obs[mask]
            try:
                if residual_cnn is not None:
                    mu, std = cnn_correct_gp(residual_cnn, gp, X_rest)
                else:
                    mu, std = gp.predict(X_rest, return_std=True)
                ucb      = mu + kappa * std
                best_idx = np.argmax(y_rest)
                rank     = np.sum(ucb > ucb[best_idx])   # 0 = best ranked 1st
                quartile = n * 0.25
                if rank <= quartile:
                    hits += 1
            except Exception:
                pass
        return hits / n

    def objective(trial):
        kappa      = trial.suggest_float('kappa',      0.5,  10.0)
        sv_weight  = trial.suggest_float('sv_weight',  0.0,  0.8)
        # prescreen_k affects speed; include it so HPO can advise on it too
        prescreen_k = trial.suggest_int('prescreen_k', 1000, 20000, step=500)
        score = loo_rank_score(kappa, sv_weight)
        return score

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=seed)
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    bp = study.best_params
    print(f"  [HPO #6] kappa={bp['kappa']:.3f}  sv_weight={bp['sv_weight']:.3f}  "
          f"prescreen_k={bp['prescreen_k']}  "
          f"LOO-rank={study.best_value:.3f}  (over {n_trials} trials)")

    return bp['kappa'], bp['sv_weight'], bp['prescreen_k'], study

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 6: MAIN BBO PIPELINE  (HPO-enhanced)
# ══════════════════════════════════════════════════════════════════════════════

def normalize(x):
    r = x.max() - x.min()
    return (x - x.min()) / r if r > 0 else np.zeros_like(x)


def hpo_enhanced_bbo(
    X_obs, y_obs, bounds,
    # ── Fallback defaults (used when HPO is disabled) ──
    kappa=4.0,
    sv_weight=0.4,
    n_random_samples=100_000,
    cnn_prescreen_k=5000,
    use_gradient_enhancement=True,
    use_cnn_amortisation=True,
    use_residual_correction=True,
    # ── HPO control flags ─────────────────────────────
    use_gp_hpo=True,           # Enhancement #8
    use_cnn_hpo=True,          # Enhancement #7
    use_acquisition_hpo=True,  # Enhancement #6
    n_gp_hpo_trials=30,
    n_cnn_hpo_trials=20,
    n_acq_hpo_trials=40,
    cnn_epochs=250,            # fallback if CNN HPO is disabled
    fn_label='F?',
):
    """
    Full BBO7 pipeline:

      STEP 1  Kernel + GP Noise HPO          (Enhancement #8, Optuna TPE)
      STEP 2  Gradient Enhancement           (BBO6 unchanged)
      STEP 3  CNN Residual Corrector HPO     (Enhancement #7, Optuna CMA-ES)
      STEP 4  CNN Gradient Predictor HPO     (Enhancement #7, Optuna CMA-ES)
      STEP 5  Acquisition HPO                (Enhancement #6, Optuna TPE)
      STEP 6  SV-like point analysis         (BBO6 unchanged)
      STEP 7  CNN pre-screen + corrected UCB (BBO6 unchanged)
      STEP 8  L-BFGS-B local refinement      (BBO6 unchanged)
    """
    dim   = len(bounds)
    lows  = np.array([b[0] for b in bounds])
    highs = np.array([b[1] for b in bounds])
    eps   = 0.01

    print('=' * 70)
    print(f'HPO-ENHANCED GP BBO  —  {fn_label}  ({dim}D, {len(y_obs)} obs)')
    print('=' * 70)

    # ── STEP 1: Kernel Optimisation + GP Noise HPO ────────────────────────────
    print('\n── STEP 1: KERNEL + GP NOISE HPO  [Enhancement #8]')
    if use_gp_hpo:
        best_name, best_kernel, best_gp_base, best_alpha, best_restarts, study_gp = \
            select_best_kernel_with_hpo(X_obs, y_obs, dim, n_trials=n_gp_hpo_trials)
    else:
        # BBO6 fallback
        from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
        kernels = {
            'RBF_ARD':        ConstantKernel(1.0,(0.1,10))*RBF(np.ones(dim),(0.01,20))+WhiteKernel(1e-5,(1e-10,1e-2)),
            'Matern_2.5_ARD': ConstantKernel(1.0,(0.1,10))*Matern(np.ones(dim),(0.01,20),nu=2.5)+WhiteKernel(1e-5,(1e-10,1e-2)),
            'Matern_1.5_ARD': ConstantKernel(1.0,(0.1,10))*Matern(np.ones(dim),(0.01,20),nu=1.5)+WhiteKernel(1e-5,(1e-10,1e-2)),
        }
        best_name, best_kernel, best_gp_base, best_score = None, None, None, -np.inf
        for name, kernel in kernels.items():
            try:
                gp_test = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=20, normalize_y=True)
                if len(X_obs) >= 5:
                    score = cross_val_score(gp_test, X_obs, y_obs, cv=min(5,len(X_obs)), scoring='neg_mean_squared_error').mean()
                else:
                    gp_test.fit(X_obs, y_obs); score = gp_test.score(X_obs, y_obs)
                gp_test.fit(X_obs, y_obs)
                if score > best_score:
                    best_score, best_name, best_kernel, best_gp_base = score, name, gp_test.kernel_, gp_test
            except Exception:
                pass
        best_alpha, best_restarts = 1e-6, 10
        print(f'  Best kernel: {best_name}')

    # ── STEP 2: Gradient Enhancement ──────────────────────────────────────────
    print('\n── STEP 2: GRADIENT ENHANCEMENT')
    gradients_est = None
    grad_mags_obs = np.zeros(len(y_obs))

    if use_gradient_enhancement:
        gradients_est = estimate_gradients_numerically(
            best_gp_base, X_obs, epsilon=eps, bounds=bounds
        )
        grad_mags_obs = np.linalg.norm(gradients_est, axis=1)
        print(f'  Gradient magnitudes — mean={grad_mags_obs.mean():.4f}  '
              f'min={grad_mags_obs.min():.4f}  max={grad_mags_obs.max():.4f}')

        gp_enhanced = GradientEnhancedGP(
            kernel=best_kernel, alpha=best_alpha, normalize_y=True
        )
        gp_enhanced.fit(X_obs, y_obs, gradients=gradients_est,
                        n_restarts=best_restarts)
        gp = gp_enhanced
    else:
        gp = best_gp_base

    # ── STEP 3: Train CNN Residual Corrector  (HPO or fallback) ───────────────
    residual_cnn = None
    if use_residual_correction and len(X_obs) >= 5:
        print('\n── STEP 3: CNN RESIDUAL CORRECTOR  [Enhancements #5 + #7]')
        print('  Computing LOO residuals from base GP...')
        residuals, loo_stds = compute_loo_residuals(best_gp_base, X_obs, y_obs)
        print(f'  LOO residuals — mean={residuals.mean():.4f}  '
              f'std={residuals.std():.4f}  max_abs={np.abs(residuals).max():.4f}')

        if use_cnn_hpo:
            residual_cnn, res_losses, res_study = tune_and_train_residual_cnn(
                X_obs, residuals, loo_stds, y_obs, dim,
                n_hpo_trials=n_cnn_hpo_trials, verbose=True
            )
        else:
            # BBO6 fallback path
            y_std    = y_obs.std() if y_obs.std() > 0 else 1.0
            res_norm = residuals / y_std
            unc_scale = np.clip(np.abs(residuals) / (loo_stds + 1e-8), 0.1, 5.0)
            targets   = np.stack([res_norm, unc_scale], axis=1)
            X_t, tgt_t = _build_augmented_tensors(X_obs, targets, n_aug=30)
            residual_cnn = ResidualCorrectorCNN(dim).to(DEVICE)
            def res_loss(p, t): return nn.MSELoss()(p[:,0:1],t[:,0:1]) + 0.5*nn.MSELoss()(p[:,1:2],t[:,1:2])
            losses = _train_one_cnn(residual_cnn, X_t, tgt_t, 1e-3, cnn_epochs, res_loss)
            residual_cnn.y_std = y_std
            residual_cnn.eval()
            print(f'  ResidualCNN trained: final loss = {losses[-1]:.6f}')
    else:
        print('\n── STEP 3: Residual correction skipped (need ≥5 obs)')

    # ── STEP 4: Train Gradient Predictor CNN  (HPO or fallback) ───────────────
    grad_cnn = None
    if use_cnn_amortisation and gradients_est is not None:
        print('\n── STEP 4: CNN GRADIENT PREDICTOR  [Enhancements #4 + #7]')
        if use_cnn_hpo:
            grad_cnn, grad_losses, grad_study = tune_and_train_gradient_cnn(
                X_obs, grad_mags_obs, dim,
                n_hpo_trials=n_cnn_hpo_trials, verbose=True
            )
        else:
            g_min, g_max = grad_mags_obs.min(), grad_mags_obs.max()
            g_range = g_max - g_min if g_max > g_min else 1.0
            grad_norm_t = (grad_mags_obs - g_min) / g_range
            X_t, g_t = _build_augmented_tensors(X_obs, grad_norm_t, n_aug=20)
            grad_cnn = GradientPredictorCNN(dim).to(DEVICE)
            losses   = _train_one_cnn(grad_cnn, X_t, g_t, 1e-3, cnn_epochs, nn.MSELoss())
            grad_cnn.g_min = g_min; grad_cnn.g_range = g_range
            grad_cnn.eval()
            print(f'  GradCNN trained: final loss = {losses[-1]:.6f}')

    # ── STEP 5: Acquisition HPO  (Enhancement #6) ─────────────────────────────
    print('\n── STEP 5: ACQUISITION PARAMETER HPO  [Enhancement #6]')
    if use_acquisition_hpo:
        kappa, sv_weight, cnn_prescreen_k, study_acq = tune_acquisition_params(
            gp, X_obs, y_obs,
            residual_cnn=residual_cnn,
            n_trials=n_acq_hpo_trials
        )
    else:
        print(f'  Using defaults: kappa={kappa}  sv_weight={sv_weight}  prescreen_k={cnn_prescreen_k}')

    # ── STEP 6: SV-like Point Analysis  (BBO6 unchanged) ─────────────────────
    print('\n── STEP 6: SV-LIKE POINT ANALYSIS')
    sv_scores  = normalize(grad_mags_obs) if gradients_est is not None else np.zeros(len(y_obs))
    top_sv_idx = np.argsort(sv_scores)[::-1][:min(5, len(y_obs))]
    print(f'  Top {len(top_sv_idx)} SV-like points:')
    for rank, idx in enumerate(top_sv_idx):
        print(f'    Rank {rank+1}: P{idx}  val={y_obs[idx]:.4f}  '
              f'grad={grad_mags_obs[idx]:.4f}  sv={sv_scores[idx]:.4f}')

    # ── STEP 7: CNN Pre-Screening + Corrected UCB  (BBO6 unchanged) ───────────
    print(f'\n── STEP 7: ACQUISITION  '
          f'(CNN pre-screen {n_random_samples:,} → {cnn_prescreen_k:,} → full GP)')

    X_candidates = np.random.uniform(low=lows, high=highs, size=(n_random_samples, dim))

    if grad_cnn is not None:
        print('  CNN pre-screening candidates...')
        cnn_grad_scores = cnn_predict_gradients(grad_cnn, X_candidates)
        cnn_ucb_quick   = normalize(cnn_grad_scores)
        top_k_idx       = np.argsort(cnn_ucb_quick)[::-1][:cnn_prescreen_k]
        X_shortlisted   = X_candidates[top_k_idx]
        print(f'  CNN shortlisted {cnn_prescreen_k:,} candidates '
              f'(saved {n_random_samples - cnn_prescreen_k:,} GP evaluations)')
    else:
        X_shortlisted   = X_candidates
        cnn_grad_scores = np.zeros(n_random_samples)
        top_k_idx       = np.arange(n_random_samples)

    print(f'  Running GP on {len(X_shortlisted):,} shortlisted candidates...')
    if residual_cnn is not None:
        mu_cand, std_cand = cnn_correct_gp(residual_cnn, gp, X_shortlisted)
        print('  Applied CNN residual correction to GP predictions')
    else:
        mu_cand, std_cand = gp.predict(X_shortlisted, return_std=True)

    grad_shortlisted = cnn_grad_scores[top_k_idx] if grad_cnn is not None else np.zeros(len(X_shortlisted))

    ucb_scores   = mu_cand + kappa * std_cand
    ucb_norm     = normalize(ucb_scores)
    grad_norm_sl = normalize(grad_shortlisted)
    sv_ucb       = (1 - sv_weight) * ucb_norm + sv_weight * grad_norm_sl

    best_idx   = np.argmax(sv_ucb)
    next_point = X_shortlisted[best_idx].copy()

    # ── STEP 8: L-BFGS-B Local Refinement  (BBO6 unchanged) ──────────────────
    print('\n── STEP 8: LOCAL REFINEMENT (L-BFGS-B)')

    def negative_sv_ucb(x):
        x2d  = np.atleast_2d(x)
        if residual_cnn is not None:
            mu, std = cnn_correct_gp(residual_cnn, gp, x2d)
        else:
            mu, std = gp.predict(x2d, return_std=True)
        grad_x = 0.0
        for d in range(dim):
            xp = x2d.copy(); xp[0, d] = min(xp[0, d] + eps, highs[d])
            xm = x2d.copy(); xm[0, d] = max(xm[0, d] - eps, lows[d])
            if residual_cnn is not None:
                mp, _ = cnn_correct_gp(residual_cnn, gp, xp)
                mm, _ = cnn_correct_gp(residual_cnn, gp, xm)
            else:
                mp = gp.predict(xp); mm = gp.predict(xm)
            grad_x += ((mp[0] - mm[0]) / (2 * eps)) ** 2
        ucb_val = mu[0] + kappa * std[0]
        sv_val  = (1 - sv_weight) * ucb_val + sv_weight * np.sqrt(grad_x)
        return -sv_val

    result = minimize(negative_sv_ucb, x0=next_point, method='L-BFGS-B', bounds=bounds)
    if result.success and -result.fun > sv_ucb[best_idx]:
        next_point = result.x
        print('  ✓ Refinement improved the acquisition score')
    else:
        print('  ○ Random search candidate retained')

    # ── Final prediction ──────────────────────────────────────────────────────
    if residual_cnn is not None:
        mu_f, std_f = cnn_correct_gp(residual_cnn, gp, next_point.reshape(1, -1))
    else:
        mu_f, std_f = gp.predict(next_point.reshape(1, -1), return_std=True)
    mu_f, std_f = float(mu_f[0]), float(std_f[0])

    # ── Summary ───────────────────────────────────────────────────────────────
    best_obs_idx = np.argmax(y_obs)
    print(f'\n{"=" * 70}')
    print(f'RESULTS — {fn_label}')
    print(f'{"=" * 70}')
    print(f'  Kernel:              {best_name}')
    print(f'  GP alpha (HPO):      {best_alpha:.2e}')
    print(f'  Gradient enhancement: {"YES" if use_gradient_enhancement else "NO"}')
    print(f'  CNN amortisation:    {"YES" if grad_cnn is not None else "NO"}')
    print(f'  CNN residual corr:   {"YES" if residual_cnn is not None else "NO"}')
    print(f'  kappa (HPO):         {kappa:.3f}')
    print(f'  sv_weight (HPO):     {sv_weight:.3f}')
    print(f'  prescreen_k (HPO):   {cnn_prescreen_k}')
    print(f'\n  Next point to sample:')
    for d, v in enumerate(next_point):
        print(f'    x{d}: {v:.6f}')
    print(f'\n  Expected performance:')
    print(f'    Mean:         {mu_f:.6f}')
    print(f'    Uncertainty:  {std_f:.6f}')
    print(f'    UCB:          {mu_f + kappa * std_f:.6f}')
    print(f'\n  Current best:  {y_obs[best_obs_idx]:.6f}  at {X_obs[best_obs_idx]}')
    improvement = (mu_f + kappa * std_f) - y_obs[best_obs_idx]
    print(f'  Potential improvement: {improvement:+.6f}')

    return {
        'fn_label':          fn_label,
        'next_point':        next_point,
        'mean':              mu_f,
        'std':               std_f,
        'ucb':               mu_f + kappa * std_f,
        'kernel':            best_name,
        'kappa':             kappa,
        'sv_weight':         sv_weight,
        'prescreen_k':       cnn_prescreen_k,
        'gp_alpha':          best_alpha,
        'gp':                gp,
        'grad_cnn':          grad_cnn,
        'residual_cnn':      residual_cnn,
        'sv_scores':         sv_scores,
        'top_sv_indices':    top_sv_idx,
        'gradients':         gradients_est,
        'improvement':       improvement,
        # ── Optuna study objects for visualisation ──
        'study_gp':          study_gp   if 'study_gp'   in dir() else None,
        'study_grad_cnn':    grad_study if 'grad_study' in dir() else None,
        'study_res_cnn':     res_study  if 'res_study'  in dir() else None,
        'study_acq':         study_acq  if 'study_acq'  in dir() else None,
    }

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# PART 8: MAIN EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

def run_all_functions(
    functions_to_run=None,
    # CNN enhancement flags
    use_cnn_amortisation=True,
    use_residual_correction=True,
    # HPO flags  ← NEW in BBO7
    use_gp_hpo=True,
    use_cnn_hpo=True,
    use_acquisition_hpo=True,
    # HPO budget
    n_gp_hpo_trials=30,
    n_cnn_hpo_trials=20,
    n_acq_hpo_trials=40,
    # Fallback defaults (used when HPO is disabled)
    kappa=4.0,
    sv_weight=0.4,
    cnn_prescreen_k=5000,
    cnn_epochs=250,
):
    """
    Run HPO-enhanced BBO for F1-F8 (or a subset).

    HPO budget guidance:
      n_gp_hpo_trials=30   → ~30 GP fits; ~2-5 min per function
      n_cnn_hpo_trials=20  → 20 CNN training runs; fast with pruning
      n_acq_hpo_trials=40  → 40 LOO evaluations; negligible cost

    To replicate BBO6 behaviour exactly, set all use_*_hpo=False.
    """
    if functions_to_run is None:
        functions_to_run = list(range(1, 9))

    print('\n' + '=' * 70)
    print('BBO7: HPO-ENHANCED BBO  —  F1-F8')
    print(f'  Enhancement #6 (Acquisition HPO):    {use_acquisition_hpo}  '
          f'({n_acq_hpo_trials} trials)')
    print(f'  Enhancement #7 (CNN Architecture HPO): {use_cnn_hpo}  '
          f'({n_cnn_hpo_trials} trials)')
    print(f'  Enhancement #8 (GP Noise HPO):        {use_gp_hpo}  '
          f'({n_gp_hpo_trials} trials)')
    print('=' * 70)

    all_results = {}

    for fn in functions_to_run:
        fn_label = f'F{fn}'
        print(f'\n\n{"#" * 70}')
        print(f'# FUNCTION {fn_label}')
        print(f'{"#" * 70}\n')

        try:
            X_obs = np.atleast_2d(np.load(f'f{fn}initial_inputs.npy'))
            y_obs = np.asarray(np.load(f'f{fn}initial_outputs.npy')).ravel()
            dim   = X_obs.shape[1]
            bounds = [(0.0, 1.0)] * dim
            print(f'Loaded: {len(y_obs)} observations, {dim}D')

            results = hpo_enhanced_bbo(
                X_obs, y_obs, bounds,
                kappa=kappa,
                sv_weight=sv_weight,
                n_random_samples=100_000,
                cnn_prescreen_k=cnn_prescreen_k,
                use_gradient_enhancement=True,
                use_cnn_amortisation=use_cnn_amortisation,
                use_residual_correction=use_residual_correction,
                use_gp_hpo=use_gp_hpo,
                use_cnn_hpo=use_cnn_hpo,
                use_acquisition_hpo=use_acquisition_hpo,
                n_gp_hpo_trials=n_gp_hpo_trials,
                n_cnn_hpo_trials=n_cnn_hpo_trials,
                n_acq_hpo_trials=n_acq_hpo_trials,
                cnn_epochs=cnn_epochs,
                fn_label=fn_label,
            )

            all_results[fn_label] = results
            np.save(f'f{fn}_next_point_bbo7.npy', results['next_point'])
            print(f'\n✓ {fn_label} complete — next point saved')

        except FileNotFoundError:
            print(f'\n✗ {fn_label}: data files not found — skipping')
        except Exception as e:
            import traceback
            print(f'\n✗ {fn_label}: failed — {e}')
            traceback.print_exc()

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f'\n\n{"=" * 80}')
    print('SUMMARY  (BBO7 HPO-Enhanced)')
    print(f'{"=" * 80}')
    header = (f'{"Func":<6} {"Dim":<4} {"Kernel":<18} {"kappa":<7} '
              f'{"sv_w":<6} {"α":<9} {"UCB":<12} {"Mean":<12} {"Std":<10}')
    print(header)
    print('-' * 80)
    for fn_label, res in sorted(all_results.items()):
        dim_str = len(res['next_point'])
        print(f"{fn_label:<6} {dim_str:<4} {res['kernel']:<18} "
              f"{res.get('kappa',4.0):<7.3f} {res.get('sv_weight',0.4):<6.3f} "
              f"{res.get('gp_alpha',1e-6):<9.2e} "
              f"{res['ucb']:<12.4f} {res['mean']:<12.4f} {res['std']:<10.4f}")

    print(f'\n{"=" * 80}')
    print('NEXT QUERY POINTS')
    print(f'{"=" * 80}')
    for fn_label, res in sorted(all_results.items()):
        coords = ', '.join([f'{v:.6f}' for v in res['next_point']])
        print(f'\n{fn_label}: [{coords}]')

    return all_results


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    results = run_all_functions(
        functions_to_run=list(range(1, 9)),
        use_cnn_amortisation=True,
        use_residual_correction=True,
        # ── HPO flags ──
        use_gp_hpo=True,            # Enhancement #8
        use_cnn_hpo=True,           # Enhancement #7
        use_acquisition_hpo=True,   # Enhancement #6
        # ── HPO budgets (reduce for faster runs) ──
        n_gp_hpo_trials=30,
        n_cnn_hpo_trials=20,
        n_acq_hpo_trials=40,
        # ── Fallback defaults (used when HPO flags are False) ──
        kappa=4.0,
        sv_weight=0.4,
        cnn_prescreen_k=5000,
        cnn_epochs=250,
    )

    print('\n✓ ALL FUNCTIONS PROCESSED')
    print('Next points saved as: f1_next_point_bbo7.npy, etc.')


BBO7: HPO-ENHANCED BBO  —  F1-F8
  Enhancement #6 (Acquisition HPO):    True  (40 trials)
  Enhancement #7 (CNN Architecture HPO): True  (20 trials)
  Enhancement #8 (GP Noise HPO):        True  (30 trials)


######################################################################
# FUNCTION F1
######################################################################

Loaded: 19 observations, 2D
HPO-ENHANCED GP BBO  —  F1  (2D, 19 obs)

── STEP 1: KERNEL + GP NOISE HPO  [Enhancement #8]
  [HPO #8] Best kernel: Matern_1.5_ARD  alpha=1.59e-06  n_restarts=8  noise_upper=3.06e-03  CV=-0.0000  (over 30 trials)

── STEP 2: GRADIENT ENHANCEMENT
  Gradient magnitudes — mean=0.0029  min=0.0001  max=0.0251
  Gradient-enhanced GP: 19 obs → 95 effective points

── STEP 3: CNN RESIDUAL CORRECTOR  [Enhancements #5 + #7]
  Computing LOO residuals from base GP...
  LOO residuals — mean=0.0002  std=0.0012  max_abs=0.0036
  [HPO #7 ResCNN] hidden=128  dropout=0.05  lr=4.94e-03  epochs=374  n_aug=37  best_lo